# BIFROST

In [ ]:
import scipp as sc
import tof

import scippnexus as snx
from ess.reduce.unwrap import GenericUnwrapWorkflow
from ess.reduce.nexus.types import *
from ess.reduce.unwrap.types import *
from ess.reduce.unwrap.lut import LtotalRange, ChopperFrameSequence

## Chopper parameters

In [ ]:
choppers = {
    "bandwidth_chopper_1": {
        "frequency": {"value": 14.0, "unit": "Hz"},
        "open": {"value": [-80.5], "unit": "deg"},
        "close": {"value": [80.5], "unit": "deg"},
        "distance": {"value": -84.01844459620825, "unit": "m"},
        "phase": {"value": 341.5771484375, "unit": "deg"},
        "type": "chopper",
        "direction": "anticlockwise",
        "name": "bandwidth_chopper_1",
    },
    "bandwidth_chopper_2": {
        "frequency": {"value": 14.0, "unit": "Hz"},
        "open": {"value": [-80.5], "unit": "deg"},
        "close": {"value": [80.5], "unit": "deg"},
        "distance": {"value": -83.98344464678199, "unit": "m"},
        "phase": {"value": 341.5771484375, "unit": "deg"},
        "type": "chopper",
        "direction": "clockwise",
        "name": "bandwidth_chopper_2",
    },
    "frame_overlap_chopper_1": {
        "frequency": {"value": 14.0, "unit": "Hz"},
        "open": {"value": [-19.13], "unit": "deg"},
        "close": {"value": [19.13], "unit": "deg"},
        "distance": {"value": -153.47749555228876, "unit": "m"},
        "phase": {"value": 44.67131423950195, "unit": "deg"},
        "type": "chopper",
        "direction": "anticlockwise",
        "name": "frame_overlap_chopper_1",
    },
    "frame_overlap_chopper_2": {
        "frequency": {"value": 14.0, "unit": "Hz"},
        "open": {"value": [-26.005], "unit": "deg"},
        "close": {"value": [26.005], "unit": "deg"},
        "distance": {"value": -147.02860013489428, "unit": "m"},
        "phase": {"value": 72.2078628540039, "unit": "deg"},
        "type": "chopper",
        "direction": "anticlockwise",
        "name": "frame_overlap_chopper_2",
    },
    "pulse_shaping_chopper_1": {
        "frequency": {"value": 196.0, "unit": "Hz"},
        "open": {"value": [-85.0], "unit": "deg"},
        "close": {"value": [85.0], "unit": "deg"},
        "distance": {"value": -155.67417487389721, "unit": "m"},
        "phase": {"value": 413.00921630859375, "unit": "deg"},
        "type": "chopper",
        "direction": "anticlockwise",
        "name": "pulse_shaping_chopper_1",
    },
    "pulse_shaping_chopper_2": {
        "frequency": {"value": 196.0, "unit": "Hz"},
        "open": {"value": [-85.0], "unit": "deg"},
        "close": {"value": [85.0], "unit": "deg"},
        "distance": {"value": -155.62517721431513, "unit": "m"},
        "phase": {"value": 575.9532470703125, "unit": "deg"},
        "type": "chopper",
        "direction": "anticlockwise",
        "name": "pulse_shaping_chopper_2",
    },
}

In [ ]:
bifrost_choppers = {}
for key, ch in choppers.items():
    bifrost_choppers[key] = tof.Chopper.from_json(name=key, params=ch).to_diskchopper()

for key, ch in bifrost_choppers.items():
    print(key)
    display(ch)

## Tof model

In [ ]:
source_position = sc.vector([0, 0, -1.61998772e02], unit="m")
source = tof.Source(facility="ess", neutrons=1_000_000, pulses=2)
source._distance = source_position.fields.z
detector = tof.Detector(distance=sc.scalar(0.0, unit="m"), name="detector")

params = [
    tof.Chopper.from_diskchopper(ch, name=key) for key, ch in bifrost_choppers.items()
] + [detector]

model = tof.Model(source=source, components=params)
res = model.run()
res.plot()

In [ ]:
res["detector"].plot()

## Wavelength lookup table

In [ ]:
wf = GenericUnwrapWorkflow(
    run_types=[SampleRun], monitor_types=[], wavelength_from="analytical"
)

wf[DiskChoppers[SampleRun]] = bifrost_choppers
wf[LtotalRange[SampleRun, snx.NXdetector]] = (
    sc.scalar(5.0, unit="m"),
    detector.distance - source_position.fields.z,
)
wf[Position[snx.NXsource, SampleRun]] = source_position

table = wf.compute(LookupTable[SampleRun, snx.NXdetector])
table.plot() + table.array['distance', -1].plot(errorbars='band')

In [ ]:
frames = wf.compute(ChopperFrameSequence[SampleRun])
at_sample = frames.propagate_to(detector.distance)
at_sample.draw()